<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'><a href="https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data/data">Pinguins - Archipel Palmer (Antarctica)</a></div>
     </th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/palmer_pinguins.png" width="96"></th>
 </tr>
</table>

<div style='text-align: center'>
<img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/penguins_species.png" width="512">
</div>


<div style='padding:15px;color:#030aa7;font-size:100%;text-align: left;font-family: Georgia, serif'><a href="https://github.com/allisonhorst/palmerpenguins/blob/main/README.md">Veuillez vous référer à la page Github officielle pour plus de détails.</a></div>

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Introduction</div></b>
## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Import libriries </div></b>

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, warnings, os
from datetime import datetime as dt
from matplotlib import pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patheffects as path_effects

import plotly.express as px
import plotly.graph_objs as go

font1 = fm.FontProperties(size=20)
font2 = fm.FontProperties(size=24)

warnings.filterwarnings(action="ignore")

if int(str(sns.__version__).split('.')[1]) > 8 : 
    plt.style.use('seaborn-v0_8-darkgrid')
else:
    plt.style.use('seaborn-darkgrid')
sns.set(font_scale=3)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Outils du document</div></b>

In [ ]:
palette = [ "#030aa7", "#e50000", "#d8863b", "#005f6a", "#6b7c85", "#751973", 
            "#0485d1", "#ff7855", "#fbeeac", "#0cb577", "#95a3a6", "#c071fe", 
            "#d1e5f0", "#fddbc7", "#ffffcb", "#12e193", "#d8dcd6", "#dfc5fe", 
          ]
sns.palplot(sns.color_palette(palette))

In [ ]:
repertoireRacine  = '.'
nomProjet         = 'Pinguins-Palmer Archipelago'

repertoireProjet  = os.path.join(repertoireRacine, nomProjet)
repertoireDonnees = os.path.join(repertoireProjet, 'repertoire.donnees')
repertoireImages  = os.path.join(repertoireProjet, 'repertoire.images')


def controleExistenceRepertoire( repertoire, create_if_needed=True):
    """Voir si le répertoire existe. S'il n'existe pas il est créé."""
    path_exists = os.path.exists(repertoire)
    if path_exists:
        if not os.path.isdir(repertoire):
            raise Exception("Trouvé le nom  "+repertoire +" mais c'est un fichier, pas un répertoire")
            # return False
        return True
    if create_if_needed:
        os.makedirs(repertoire)

def sauvegarderImage( fichier):
    """Enregistrez la figure. Appelez la méthode juste avant plt.show ()."""
    controleExistenceRepertoire(repertoireImages)
    plt.savefig(os.path.join(repertoireImages,
                             fichier+f"--{dt.now().strftime('%Y_%m_%d_%H.%M.%S')}.png"), 
                             dpi=600, 
                             bbox_inches='tight')

def sauvegarderImageSNS( sns_plot, fichier):
    """Enregistrez la figure. Appelez la méthode juste avant plt.show ()."""
    controleExistenceRepertoire(repertoireImages)
    fig = sns_plot.get_figure()
    fig.savefig(os.path.join(repertoireImages,fichier+'.png'))
    
controleExistenceRepertoire(repertoireProjet);
controleExistenceRepertoire(repertoireDonnees);
controleExistenceRepertoire(repertoireImages);

In [ ]:
def formatPct(pct, allvals):
    total = int(round(pct/100. * np.sum(allvals)))
    return "{:.2f}%\n({:d})".format(pct, total)    

In [ ]:
def affichageDistribution(colonne,couleur,ax, nom=''):
    graph = sns.distplot(colonne, color=couleur, ax=ax)
    graph.set(ylabel=None)
    moyenne, mediane = float(colonne.mean()), \
                   float(colonne.median())
    
    ax.axvline(moyenne, color='g', linestyle='-', label=f"{nom:12s} mean   = {moyenne:0.4f}", lw=2)
    ax.axvline(mediane, color='b', linestyle='--', label=f"{nom:12s} median = {mediane:0.4f}", lw=2)
    graph.legend(loc="upper right")
    

In [ ]:
def afficheDendrogram(*args, **kwargs):
    max_d = kwargs.pop('max_d', None)
    if max_d and 'color_threshold' not in kwargs:
        kwargs['color_threshold'] = max_d
    annotate_above = kwargs.pop('annotate_above', 0)

    ddata = dendrogram(*args, **kwargs)

    if not kwargs.get('no_plot', False):
        plt.title('Classification Hiérarchique Ascendante')
        plt.xlabel('Villes ou (taille du cluster)')
        plt.ylabel('Distance')
        for i, d, c in zip(ddata['icoord'], ddata['dcoord'], ddata['color_list']):
            x = 0.5 * sum(i[1:3])
            y = d[1]
            if y > annotate_above:
                plt.plot(x, y, 'o', c=c)
                plt.annotate("%.3g" % y, (x, y), xytext=(0, -5),
                             textcoords='offset points',
                             va='top', ha='center')
        if max_d:
            plt.axhline(y=max_d, c='k')
    return ddata

In [ ]:
def add_median_labels(ax, precision='.1f'):
    lines = ax.get_lines()
    # determine number of lines per box (this varies with/without fliers)
    boxes = [c for c in ax.get_children() if type(c).__name__ == 'PathPatch']
    lines_per_box = int(len(lines) / len(boxes))
    # iterate over median lines
    for median in lines[4:len(lines):lines_per_box]:
        # display median value at center of median line
        x, y = (data.mean() for data in median.get_data())
        # choose value depending on horizontal or vertical plot orientation
        value = x if (median.get_xdata()[1]-median.get_xdata()[0]) == 0 else y
        text = ax.text(
                       x, 
                       y, 
                       f'{value:{precision}}', 
                       verticalalignment='center',
                       horizontalalignment='center', 
                       fontweight='bold', 
                       color='black',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.6),
                      )
        # créer une bordure de couleur médiane autour du texte blanc pour le contraste 
        text.set_path_effects([
            path_effects.Stroke(linewidth=3, foreground=median.get_color()),
            path_effects.Normal(),
        ])   

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture des données</div></b>

<table>
    <CAPTION style='padding:15px;color:#030aa7;font-size:150%;text-align: left;font-weight: bold;font-family: Georgia, serif'>penguins_size.csv</CAPTION>    
<tr>                                                                                   
     <th>
            <table>
            <tr>                                                                                   
                 <th  style="text-align:left;background-color:#053061;color:white;">Colonne initiale </th>
                 <th  style="text-align:left;background-color:#053061;color:white;">Description</th>
             </tr>
            <tr>
                <th  style="text-align:left">species</th>
                <th  style="text-align:left">espèces de manchots (Chinstrap, Adélie ou Gentoo)</th>
            </tr>
            <tr>
                <th  style="text-align:left">culmen_length_mm</th>
                <th  style="text-align:left">longueur du culmen (culmen est la crête supérieure du bec d'un oiseau)(mm)</th>
            </tr>
            <tr>
                <th  style="text-align:left">culmen_depth_mm</th>
                <th  style="text-align:left">profondeur du culmen (culmen est la crête supérieure du bec d'un oiseau)(mm)</th>
            </tr>
            <tr>
                <th  style="text-align:left">flipper_length_mm</th>
                <th  style="text-align:left">longueur des nageoires (mm)</th>
            </tr>
            <tr>
                <th  style="text-align:left">body_mass_g</th>
                <th  style="text-align:left">masse corporelle (g)</th>
            </tr>
            <tr>
                <th  style="text-align:left">island</th>
                <th  style="text-align:left">nom de l'île (Dream, Torgersen ou Biscoe) dans l'archipel Palmer (Antarctique)</th>
            </tr>
            <tr>
                <th  style="text-align:left">sex</th>
                <th  style="text-align:left">sexe du manchot</th>
            </tr>
            </table>
     </th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/culmen.png" width="512"></th>
 </tr>
</table>

In [ ]:
donnees = pd.read_csv('../donnees/Palmer Archipelago-Antarctica-Penguin/penguins_size.csv')

In [ ]:
quantitatives = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
qualitatives = ['island', 'sex']
cible = 'species'

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Effacement des valeurs non renseignés</div></b>

In [ ]:
donnees[donnees.body_mass_g.isna()]

<b><div style='padding:15px;color:#030aa7;font-size:100%;text-align: left'>Je pense que je peux supprimer les lignes avec les entrées nulles sans causer de problèmes majeurs.</div></b>

In [ ]:
donnees = donnees[~donnees.body_mass_g.isna()]
donnees.info()

In [ ]:
donnees.duplicated().sum()

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Variables qualitatives</div></b>

In [ ]:
qualitatives = ['island', 'sex']

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Espèces de manchots</div></b>

In [ ]:
donnees.species.sort_values().unique()

In [ ]:
dict_species = {nom:i for i, nom in enumerate(donnees.species.sort_values().unique())}
dictR_species = {i:nom for i, nom in enumerate(donnees.species.sort_values().unique())}
dict_species,dictR_species

In [ ]:
donnees['espece'] = donnees.species
donnees.species = donnees.species.apply(lambda x: dict_species[x])

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Sexe du manchot</div></b>

In [ ]:
donnees.sex.sort_values().unique()

In [ ]:
donnees[(donnees.sex == '.' )|(donnees.sex.isna() ) ]

In [ ]:
donnees.sex = donnees.sex.apply(lambda x : None if x == '.' else x).fillna('none').apply(lambda x: str(x).lower())
donnees.sex.sort_values().unique()

In [ ]:
dict_sex = {nom:i for i, nom in enumerate(donnees.sex.sort_values().unique())}
dictR_sex = {i:nom for i, nom in enumerate(donnees.sex.sort_values().unique())}
dict_sex,dictR_sex

In [ ]:
donnees['sexe'] = donnees.sex
donnees.sex = donnees.sex.apply(lambda x: dict_sex[x])

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Nom de l'île</div></b>

<img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/pinguins_archipel_palmer.png" width="1024">

In [ ]:
donnees.island.sort_values().unique()

In [ ]:
dict_island = {nom:i for i, nom in enumerate(donnees.island.sort_values().unique())}
dictR_island = {i:nom for i, nom in enumerate(donnees.island.sort_values().unique())}
dict_island,dictR_island

In [ ]:
donnees['nom_ile'] = donnees.island
donnees.island = donnees.island.apply(lambda x: dict_island[x])

In [ ]:
qualitativesT = ['espece', 'sexe', 'nom_ile']

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Statistiques descriptives et analyse de données</div></b>

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Couleurs variables qualitatives</div></b>

In [ ]:
couleursEspece = {nom:couleur for nom,couleur in zip(donnees.espece.sort_values().unique(),["#751973","#005f6a","#d8863b"])}
couleursSexe = {nom:couleur for nom,couleur in zip(donnees.sexe.sort_values().unique(),["#0485d1","#ff7855","#95a3a6"])}
couleursNomIle = {nom:couleur for nom,couleur in zip(donnees.nom_ile.sort_values().unique(),["#d1e5f0", "#fddbc7","#dfc5fe"])}
sns.palplot(sns.color_palette(couleursEspece.values()))
sns.palplot(sns.color_palette(couleursSexe.values()))
sns.palplot(sns.color_palette(couleursNomIle.values()))

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Statistiques  descriptives</div></b>

In [ ]:
donnees.sample(5)

In [ ]:
donnees.drop(columns=qualitativesT).describe().style.format("{:0.2f}") #.background_gradient(cmap=plt.get_cmap('Blues'),axis=0)

In [ ]:
donnees.columns

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Structure de l’échantillon des données </div></b>

In [ ]:
donnees.sample(5)

In [ ]:
affichage = donnees.groupby(['espece','sexe']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.pivot_table(
    index='espece',
    columns='sexe',
    values='nombre',
    # fill_value=0
    ).style.format("{:0.2f}").background_gradient(cmap=plt.get_cmap('Blues'),axis=0)

In [ ]:
affichage = donnees.groupby(['espece','nom_ile']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.pivot_table(
    index='espece',
    columns='nom_ile',
    values='nombre',
    # fill_value=0
    ).style.format("{:0.2f}").background_gradient(cmap=plt.get_cmap('Blues'),axis=0)

In [ ]:
affichage = donnees.groupby(['nom_ile','espece','sexe']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.pivot_table(
    index=['espece','sexe'],
    columns='nom_ile',
    values='nombre',
    # fill_value=0
    ).style.format("{:0.2f}").background_gradient(cmap=plt.get_cmap('Blues'),axis=0)

In [ ]:
radius,size=0.8,0.3
fig,(ax0,ax1,ax2) = plt.subplots(ncols=3,figsize=(92,36), subplot_kw=dict(aspect="equal"))

affichage = donnees.groupby(['espece']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

wedges, texts, autotexts =  ax0.pie(
         affichage['nombre'],
         autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
         labels=affichage['espece'].values,
         # shadow=True, 
         counterclock=False,
         startangle=0 ,
         colors = couleursEspece.values(),
         # pctdistance=0.4, 
         labeldistance=1.1, 
         textprops=dict(color="#030aa7"),
         explode=(0.03,0.03,0.03)
      );
plt.setp(autotexts, size=48, weight="bold",color="w")
plt.setp(texts, size=64, weight="bold")

affichage = donnees.groupby(['espece']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.sort_values(['espece'],inplace=True)

wedges, texts, autotexts =  ax1.pie( 
                                    affichage['nombre'],
                                    autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
                                    labels=affichage['espece'].values,
                                    counterclock=False,
                                    wedgeprops=dict(width=0.3),
                                    pctdistance=0.8,
                                    radius=radius, 
                                    colors=couleursEspece.values(),
                                    # wedgeprops=dict(width=size, edgecolor='w'),
                                    textprops=dict(color='#053061',fontsize='x-large'),
                                    startangle=0 ,)
plt.setp(autotexts, size=48, weight="bold",color="w")
plt.setp(texts, size=64, weight="bold")

affichage = donnees.groupby(['espece','sexe']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.sort_values(['espece','sexe'],inplace=True)

wedges, texts, autotexts =  ax1.pie( 
                                    affichage['nombre'],
                                    autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
                                    # labels=affichage['sexe'].values,
                                    counterclock=False,
                                    pctdistance=0.7,
                                    radius=radius-size, 
                                    colors=affichage['sexe'].apply(lambda x : couleursSexe[x]),
                                    wedgeprops=dict(width=size, edgecolor='w'),
                                    textprops=dict(color="#053061"),
                                    startangle=0 ,)
plt.setp(autotexts, size=32, weight="bold")
plt.setp(texts, size=32, weight="bold",color="w")
ax1.legend(wedges, affichage['sexe'].unique(),
          title="Sexe",
          loc="upper left",
          fontsize ='x-large', 
          title_fontsize='x-large', 
          bbox_to_anchor=(0, 1))
ax1.set_title("Sexe", size=96);

affichage = donnees.groupby(['espece']).nom_ile.count().reset_index().rename(columns={'nom_ile':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.sort_values(['espece'],inplace=True)
colorsSex = {'Biscoe':"#d1e5f0", 'Dream':"#fddbc7", 'Torgersen':"#dfc5fe"}

wedges, texts, autotexts =  ax2.pie( 
                                    affichage['nombre'],
                                    autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
                                    labels=affichage['espece'].values,
                                    counterclock=False,
                                    wedgeprops=dict(width=0.3),
                                    pctdistance=0.8,
                                    radius=radius, 
                                    colors=couleursEspece.values(),
                                    # wedgeprops=dict(width=size, edgecolor='w'),
                                    textprops=dict(color='#053061',fontsize='x-large'),
                                    startangle=0 ,)
plt.setp(autotexts, size=48, weight="bold",color="w")
plt.setp(texts, size=64, weight="bold")

affichage = donnees.groupby(['espece','nom_ile']).island.count().reset_index().rename(columns={'island':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage.sort_values(['espece','nom_ile'],inplace=True)

wedges, texts, autotexts =  ax2.pie( 
                                    affichage['nombre'],
                                    autopct=lambda pct: formatPct(pct, affichage.nombre.values),   # autopct='%1.2f%%', 
                                    # labels=affichage['nom_ile'].values,
                                    counterclock=False,
                                    pctdistance=0.7,
                                    radius=radius-size, 
                                    colors=affichage['nom_ile'].apply(lambda x : couleursNomIle[x]),
                                    wedgeprops=dict(width=size, edgecolor='w'),
                                    textprops=dict(color="#053061"),
                                    startangle=0 ,)
plt.setp(autotexts, size=32, weight="bold")
plt.setp(texts, size=32, weight="bold",color="w")
ax2.legend(wedges, affichage['nom_ile'].unique(),
          title="Nom de l’île",
          loc="upper left",
          fontsize ='x-large', 
          title_fontsize='x-large', 
          bbox_to_anchor=(0, 1))
ax2.set_title("Location géographique", size=96);
fig.suptitle("Espèces de pingouins",fontsize=128);
plt.tight_layout()
# fig.set_facecolor("#ffffcb")

In [ ]:
affichage = donnees.groupby(['nom_ile','espece','sexe']).sex.count().reset_index().rename(columns={'sex':'nombre'})
affichage['%'] = affichage.nombre * 100 / affichage.nombre.sum()
affichage

fig = go.Figure(px.treemap(affichage, 
                           path=[px.Constant("Île Anvers"), 'nom_ile', 'espece','sexe'], values='nombre',
                           color='nombre', 
                           hover_data=['nom_ile', 'espece','sexe'],
                  color_continuous_scale='RdBu',
                  color_continuous_midpoint=affichage['nombre'].mean(),
                 width=1152,
                 height=768
                ))
fig.show()

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Distribution de l’échantillon des données </div></b>

In [ ]:
donnees.sample(5)

In [ ]:
graph = sns.pairplot(
             donnees.drop(columns=qualitatives+[cible]),
             hue='espece', 
             size=16, 
             aspect=1, 
             palette=couleursEspece.values(), 
             plot_kws={"s": 1200,"alpha":0.6}, 
             markers=["o", "s", "^"],   
             # corner=True, 
             diag_kind="kde")
graph.map_upper(sns.kdeplot, levels=24, color=".2");
graph._legend.remove()
graph.add_legend(fontsize='xx-large', title_fontsize='xx-large');
# sauvegarderImage('Distribution de l’échantillon des données')

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Les trois graphiques les plus  représentatives</div></b>
Les trois graphiques les plus  représentatives  du point de vue de la variance et séparation des classes.

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(54,18));
for i,colonne in enumerate(['culmen_depth_mm','flipper_length_mm', 'body_mass_g']):
    for espece in couleursEspece:
        sns.regplot(     data=donnees[(donnees.espece==espece)&(donnees.sex==0)], 
                         x="culmen_length_mm", 
                         y=colonne,
                         color=couleursEspece[espece],
                         scatter_kws={"s": 200,"alpha":0.6}, 
                         marker="o", 
                         label=f'{espece}-female',
                         ax=axes[i])
        sns.regplot(     data=donnees[(donnees.espece==espece)&(donnees.sex==1)], 
                         x="culmen_length_mm", 
                         y=colonne,
                         color=couleursEspece[espece],
                         scatter_kws={"s": 200,"alpha":0.6}, 
                         marker="s",
                         label=f'{espece}-male',
                         ax=axes[i])
        sns.scatterplot( data=donnees[(donnees.espece==espece)&(donnees.sex==2)], 
                         x="culmen_length_mm", 
                         y=colonne,
                         color=couleursEspece[espece],
                         s=600,
                         alpha=0.8,
                         marker="*", 
                         label=f'{espece}-none',
                         legend=True,
                         ax=axes[i])
# sauvegarderImage('Les trois graphiques les plus représentatives')        

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Corrélation de Pearson</div></b>

<table>        
<tr>                                                                                   
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/rmse.png" ></th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/rse.png" ><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/correlation_pearson.png" ></th>
</tr> 
</table>
<img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/correlation_pearson_graphs.png" width="1024">

In [ ]:
plt.figure(figsize=(24,24))
sns.set(font_scale=3)
plt.title('Correlation Pearson des variables', y=1.05, size=36)
sns.heatmap(donnees[quantitatives].corr(),linewidths=0.3, fmt= '.2f', #vmax=1.0, 
            square=True, cmap='coolwarm', linecolor='white', annot=True)
# sauvegarderImage('Correlation Pearson des variables')   
sns.set(font_scale=2)

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Analyse des variables corrélés</div></b>

In [ ]:
sns.jointplot(data=donnees,
              x='flipper_length_mm',
              y='body_mass_g',
              hue='espece',
              height=10,
              ratio=5, 
              palette=couleursEspece.values());
# sauvegarderImage('Analyse des variables corrélés espèces')  

In [ ]:
sns.jointplot(data=donnees,
              x='flipper_length_mm',
              y='body_mass_g',
              kind ='reg', #{ "scatter" | "kde" | "hist" | "hex" | "reg" | "resid" }
              # hue='espece',
              height=12,
              ratio=5,);
# sauvegarderImage('Analyse des variables corrélés régression')  

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Distribution variables quantitatives</div></b>

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(36,28));
for ax,colonne in zip(axes.ravel(),quantitatives):
    affichageDistribution(donnees[colonne],"#6b7c85",ax)
# sauvegarderImage('Distribution Variables Quantitatives')

In [ ]:
donnees.sample(10)

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(36,28));
for ax,colonne in zip(axes.ravel(),quantitatives):
    affichageDistribution(donnees.loc[donnees.espece == 'Adelie',colonne],'#751973',ax,nom='Adelie')
    affichageDistribution(donnees.loc[donnees.espece == 'Chinstrap',colonne],'#005f6a',ax,nom='Chinstrap')
    affichageDistribution(donnees.loc[donnees.espece == 'Gentoo',colonne],'#d8863b',ax,nom='Gentoo') 
# sauvegarderImage('Distribution Variables Quantitatives-par espèce')

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Centrage et réduction des données</div></b>
<table>
<tr>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/moyenne.png"></th>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/ecart_type.png"></th>
<th><img src="https://raw.githubusercontent.com/rbizoi/MachineLearning/refs/heads/master/images/centrage_reduction.png"></th>
</tr>
</table>

In [ ]:
modelStd = StandardScaler()
donnees[quantitatives] = modelStd.fit_transform(donnees[quantitatives])

In [ ]:
donnees.sample(10)

In [ ]:
donnees.rename(columns={'culmen_length_mm':'culmen_length', 
                        'culmen_depth_mm':'culmen_depth',
                        'flipper_length_mm':'flipper_length', 
                        'body_mass_g':'body_mass'}, inplace=True)

In [ ]:
graph = sns.pairplot(
             donnees.drop(columns=qualitatives+[cible]),
             hue='espece', 
             size=16, 
             aspect=1, 
             palette=couleursEspece.values(), 
             plot_kws={"s": 1200,"alpha":0.6}, 
             markers=["o", "s", "^"],   
             # corner=True, 
             diag_kind="kde")
graph.map_upper(sns.kdeplot, levels=24, color=".2");
graph._legend.remove()
graph.add_legend(fontsize='xx-large', title_fontsize='xx-large');
# sauvegarderImage('Distribution de l’échantillon des données')

### <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Les trois graphiques les plus  représentatives</div></b>
Les trois graphiques les plus  représentatives  du point de vue de la variance et séparation des classes.

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(54,18));
for i,colonne in enumerate(['culmen_depth','flipper_length', 'body_mass']):
    for espece in couleursEspece:
        sns.regplot(     data=donnees[(donnees.espece==espece)&(donnees.sex==0)], 
                         x="culmen_length", 
                         y=colonne,
                         color=couleursEspece[espece],
                         scatter_kws={"s": 200,"alpha":0.6}, 
                         marker="o", 
                         label=f'{espece}-female',
                         ax=axes[i])
        sns.regplot(     data=donnees[(donnees.espece==espece)&(donnees.sex==1)], 
                         x="culmen_length", 
                         y=colonne,
                         color=couleursEspece[espece],
                         scatter_kws={"s": 200,"alpha":0.6}, 
                         marker="s",
                         label=f'{espece}-male',
                         ax=axes[i])
        sns.scatterplot( data=donnees[(donnees.espece==espece)&(donnees.sex==2)], 
                         x="culmen_length", 
                         y=colonne,
                         color=couleursEspece[espece],
                         s=600,
                         alpha=0.8,
                         marker="*", 
                         label=f'{espece}-none',
                         legend=True,
                         ax=axes[i])
# sauvegarderImage('Les trois graphiques les plus représentatives')        

In [ ]:
fig,ax = plt.subplots(1,len(quantitatives),figsize=(56,12));

for i,colonne in enumerate(['culmen_length', 'culmen_depth', 'flipper_length', 'body_mass']):
    sns.boxplot(x='sexe', 
                y=colonne,
                data=donnees,
                hue='espece', 
                palette=list(couleursEspece.values()), 
                medianprops = dict(color="white", linewidth=4, alpha=0.9),
                flierprops  = dict(markerfacecolor="#707070", marker="d"),
                fliersize   = 10,                
                ax=ax[i]);
    add_median_labels(ax[i])

# sauvegarderImage('Distribution des variables centrées et réduites par espèce-boxplot')   